## Libraries

In [1]:
# Prevent OpenMP runtime conflict between libraries like PyTorch, TensorFlow, and NumPy on Windows
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchmetrics.classification import MulticlassAccuracy

from tqdm import tqdm
import numpy as np
from prettytable import PrettyTable

from danflow.training import Trainer

## Load Data

In [2]:
data = torch.load('../saved_values/shuttle_data.pt', weights_only=False)

x_train = data["x_train"]
y_train = data["y_train"]

x_valid = data["x_valid"]
y_valid = data["y_valid"]

x_test = data["x_test"]
y_test = data["y_test"]

## Convert to Tensors

In [3]:
x_train = torch.tensor(np.asarray(x_train), dtype=torch.float32)
y_train = torch.tensor(np.asarray(y_train), dtype=torch.long)

x_valid = torch.tensor(np.asarray(x_valid), dtype=torch.float32)
y_valid = torch.tensor(np.asarray(y_valid), dtype=torch.long)

x_test = torch.tensor(np.asarray(x_test), dtype=torch.float32)
y_test = torch.tensor(np.asarray(y_test), dtype=torch.long)

## Data Loader

In [4]:
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

valid_dataset = TensorDataset(x_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=256)

test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=256)

## MLP Model

The model is a 3-layer MLP with two hidden layers containing 64 and 32 neurons, respectively, using ReLU activation functions. The output layer contains 7 neurons, corresponding to the 7 target classes.

In [5]:
def mlp_model():
    "Initializes multi layer perceptron model"
    in_features = 9
    num_class = 7
    h1 = 64
    h2 = 32

    model = nn.Sequential(nn.Linear(in_features, h1),
                           nn.ReLU(),
                           nn.Linear(h1, h2),
                           nn.ReLU(),
                           nn.Linear(h2, num_class))

    return model


model = mlp_model()
model

Sequential(
  (0): Linear(in_features=9, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=7, bias=True)
)

## Cross-Entropy Loss
$$
\mathcal{L} = -\log(\hat{y}_{\text{true}})
$$

In [6]:
loss_fn = nn.CrossEntropyLoss()

## Optimizer

In [7]:
optimizer = optim.SGD(model.parameters(),
                      lr=0.01,
                      momentum=0.9,
                      nesterov=True,
                      weight_decay=1e-4)

## Model Verification

### Step 1: Check Forward Path

Calculate loss for one batch

In [8]:
x_batch, y_batch = next(iter(train_loader))

print(f"x_batch shape = {x_batch.shape}")
print(f"y_batch shape = {y_batch.shape}")

outputs = model(x_batch)

print(f"outputs shape = {outputs.shape}")

loss = loss_fn(outputs, y_batch)
print(f"loss = {loss:.4f}")

x_batch shape = torch.Size([128, 9])
y_batch shape = torch.Size([128])
outputs shape = torch.Size([128, 7])
loss = 1.9167


### Step 2: Check Backward Path

Select 5 random batches and overfit the model

In [9]:
mini_train_dataset, _ = random_split(train_dataset, 
                                     (1000, (len(train_dataset)-1000)))

mini_loader = DataLoader(mini_train_dataset, 
                         batch_size=200, 
                         shuffle=True)

In [10]:
accuracy = MulticlassAccuracy(num_classes=7)

trainer = Trainer(model,
        optimizer,
        loss_fn,
        accuracy)

In [11]:
for epoch in range(500):
    with tqdm(total=1, desc=f"Epoch {epoch}", unit="batch") as pbar:
        loss, acc = trainer.train_epoch(mini_loader)

        pbar.set_postfix(
            accuracy=f"{acc:.4f}",
            loss=f"{loss:.4f}"
        )
        pbar.update(1)

Epoch 499: 100%|██████████| 1/1 [00:00<00:00, 22.30batch/s, accuracy=0.8750, loss=0.0088]


## Select the Best Learning Rate

In [15]:
for lr in [0.1, 0.01, 0.001, 0.0001]:
    print(f"LR={lr}")

    model = mlp_model()

    optimizer = optim.SGD(model.parameters(),
                          lr=lr,
                          weight_decay=1e-4)

    trainer = Trainer(model,
        optimizer,
        loss_fn,
        accuracy)
    for epoch in range(5):
        with tqdm(total=1, desc=f"Epoch {epoch}", unit="batch") as pbar:
                loss, acc = trainer.train_epoch(train_loader)
        
                pbar.set_postfix(
                    accuracy=f"{acc:.4f}",
                    loss=f"{loss:.4f}"
                )
                pbar.update(1)

    print()
    

LR=0.1


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.54s/batch, accuracy=0.4263, loss=0.0549]



LR=0.01


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.30s/batch, accuracy=0.3909, loss=0.2371]



LR=0.001


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.42s/batch, accuracy=0.1429, loss=1.1685]



LR=0.0001


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.24s/batch, accuracy=0.1413, loss=1.8784]

## Small Grid

In [16]:
my_table = PrettyTable(
    ["Learning Rate", "Weight decay", "Accuracy", "Loss"]
)

for lr in [0.1, 0.15, 0.20, 0.25]:
    for wd in [0.0, 1e-4, 1e-5, 1e-6]:

        model = mlp_model()
       
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            weight_decay=wd
        )

        trainer = Trainer(
            model,
            optimizer,
            loss_fn,
            accuracy
        )

        tqdm.write(f"LR={lr}, WD={wd}")

        for epoch in range(5):
            with tqdm(
                total=1,
                desc=f"Epoch {epoch}",
                unit="batch"
            ) as pbar:

                loss, acc = trainer.train_epoch(train_loader)

                pbar.set_postfix(
                    accuracy=f"{acc:.4f}",
                    loss=f"{loss:.4f}"
                )

                pbar.update(1)

        my_table.add_row([
            lr,
            wd,
            f"{100. * acc:.4f}",
            f"{loss:.4f}"
        ])

        tqdm.write("")

    my_table.add_row([
        "-" * 20,
        "-" * 20,
        "-" * 20,
        "-" * 20
    ])

print(my_table)

LR=0.1, WD=0.0


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.80s/batch, accuracy=0.4388, loss=0.1475]



LR=0.1, WD=0.0001


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.31s/batch, accuracy=0.4099, loss=0.1391]



LR=0.1, WD=1e-05


...


LR=0.25, WD=1e-05


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.20s/batch, accuracy=0.2531, loss=0.4546]



LR=0.25, WD=1e-06


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.38s/batch, accuracy=0.4235, loss=0.1988]


+----------------------+----------------------+----------------------+----------------------+
|    Learning Rate     |     Weight decay     |       Accuracy       |         Loss         |
+----------------------+----------------------+----------------------+----------------------+
|         0.1          |         0.0          |       43.8798        |        0.1475        |
|         0.1          |        0.0001        |       40.9927        |        0.1391        |
|         0.1          |        1e-05         |       42.1915        |        0.1146        |
|         0.1          |        1e-06         |       41.7575        |        0.1365        |
| -------------------- | -------------------- | -------------------- | -------------------- |
|         0.15         |         0.0          |       46.0910        |        0.0434        |
|         0.15         |        0.0001        |       49.6471        |        0.0522        |
|         0.15         |        1e-05         |       50.26